In [37]:
from datetime import date

from flights.evaluation.evaluation import main

outputs = main("algorithm", "sfdps", test_dates = date(2026,3,1), adsb_src="adsbx")

Stats for algorithm:sfdps: {'true_positive': 6716, 'false_positive': 23, 'false_negative': 16, 'precision': 0.9965870307167235, 'recall': 0.9976232917409388, 'f1': 0.9971048919902012, 'takeoff_airport_ident_match_pct': 0.9980643240023823, 'landing_airport_ident_match_pct': 0.9955330553901132, 'airport_ident_match_pct': 0.9944907683144729}
Stats CSV saved to /Volumes/T2-SSD/planequery/data/experiments/comparison/2026-07-19_18-58_823393a_flights_evaluation.csv


In [38]:
import polars as pl

In [60]:
df = outputs[0][0]
COLUMNS = ["icao", "takeoff_time", "landing_time", "takeoff_airport_ident", "landing_airport_ident", "takeoff_time_df1", "landing_time_df1", "takeoff_airport_ident_df1", "landing_airport_ident_df1", "match_status"]
df = df.select(COLUMNS)

In [61]:
df.filter(pl.col("match_status") != "both")

icao,takeoff_time,landing_time,takeoff_airport_ident,landing_airport_ident,takeoff_time_df1,landing_time_df1,takeoff_airport_ident_df1,landing_airport_ident_df1,match_status
str,datetime[ms],datetime[ms],str,str,datetime[ms],datetime[ms],str,str,str
"""a1492e""",2026-03-01 18:51:05.630,2026-03-01 20:01:11.200,"""KCRG""","""KCRG""",null,null,null,null,"""df0_only"""
"""a1a4a7""",2026-03-01 15:39:39.850,2026-03-01 17:19:10.730,"""KBHM""","""KNEW""",null,null,null,null,"""df0_only"""
"""a1a4a7""",2026-03-01 17:23:25.830,2026-03-01 19:18:10.300,"""KNEW""","""KBHM""",null,null,null,null,"""df0_only"""
"""a1a4a7""",null,null,null,null,2026-03-01 15:40:00,2026-03-01 19:20:00,"""KBHM""","""KBHM""","""df1_only"""
"""a1c2e3""",null,null,null,null,2026-03-01 18:19:00,2026-03-01 23:16:00,"""KAPF""","""KARR""","""df1_only"""
…,…,…,…,…,…,…,…,…,…
"""ad963b""",null,null,null,null,2026-03-01 19:23:00,2026-03-01 22:43:00,"""KAPF""","""KAPF""","""df1_only"""
"""ad963b""",2026-03-01 19:22:48.570,2026-03-01 21:41:18.240,"""KAPF""","""KAPF""",null,null,null,null,"""df0_only"""
"""adab24""",2026-03-01 18:11:00.830,2026-03-01 18:53:54.740,"""KPIE""","""FA71""",null,null,null,null,"""df0_only"""


In [66]:
df.filter(pl.col("match_status") != "both").get_column("icao").unique().to_list()

['a25dca',
 'a3a1dc',
 'a1e6b9',
 'adab24',
 'a81525',
 'a875ab',
 'adccd0',
 'a63c0b',
 'a1c2e3',
 'a1a4a7',
 'a5a744',
 'a32756',
 'a3e348',
 'a1d684',
 'a1492e',
 'a654ec',
 'abfc78',
 'aafb20',
 'a25ec5',
 'a6ea99',
 'ad963b',
 'acb4b2',
 'a9b2c8']

## Manual review of mismatched ICAOs

The complete ADSB-X trajectories, algorithm flights (`df0`), and SFDPS flights (`df1`) were reviewed for 2026-03-01.

### Summary

- **Gold/df1 problem:** 18 ICAOs
- **Genuine algorithm/df0 failure:** 4 ICAOs
- **Both datasets have problems:** 1 ICAO (`a654ec`)

### Mismatch error percentages

Percentages use the 23 mismatched ICAOs as the denominator.

| Test set Error (%) | Gold-Dataset Error (%) | Error in both (%) |
|---:|---:|---:|
| 17.39% | 78.26% | 4.35% |

| ICAO | Incorrect dataset | Reason |
|---|---|---|
| `a25dca` | **df1 gold** | SFDPS combines the KSUS-KUBX-KSUS activity into one KSUS-KSUS flight. |
| `a3a1dc` | **df1 gold** | The second flight actually departs KRBD around 21:10; df1 incorrectly reuses 18:11. |
| `a1e6b9` | **df1 gold** | The aircraft remains on the ground at KHOU until the real 13:37 takeoff; df1 says 12:34. |
| `adab24` | **df0 algorithm** | The algorithm invents an FA71 stop during ADS-B gaps. The aircraft remains airborne and never gets closer than about 42 km to FA71. |
| `a81525` | **df0 algorithm** | A clear KAUS ground-air-ground flight from approximately 16:56-17:21 is completely missing. |
| `a875ab` | **df1 gold** | KBKL-KLNS departs around 10:38; df1 incorrectly reuses 06:12. |
| `adccd0` | **df1 gold** | KPHL-KHPN departs around 17:43; df1 incorrectly reuses 14:03. |
| `a63c0b` | **df1 gold** | KENW-KBNA departs around 16:25; df1 incorrectly reuses 12:48. |
| `a1c2e3` | **df1 gold** | The aircraft stays on the ground at KAPF until 20:33; df1 claims an 18:19 departure. |
| `a1a4a7` | **df1 gold** | A real ground stop at KNEW separates KBHM-KNEW and KNEW-KBHM; df1 combines them. |
| `a5a744` | **df1 gold** | The trajectory supports an intermediate KM89 landing/touch-and-go; df1 combines both legs. |
| `a32756` | **df1 gold** | SFDPS omits real morning KRIC pattern activity. The algorithm segmentation is imperfect, but the flagged extra flight is real. |
| `a3e348` | **df1 gold** | FA54-KHOU actually departs around 17:57; df1 incorrectly reuses 15:07. |
| `a1d684` | **df1 gold** | An intermediate KTYR stop separates two legs; df1 combines them into KACT-KACT. |
| `a1492e` | **df1 gold** | SFDPS omits a real local KCRG flight from approximately 18:51-20:01. |
| `a654ec` | **both** | df1 omits the earlier KFFZ local flight; df0 also fails to split the later KFFZ-KPRC-KFFZ touch-and-go. |
| `abfc78` | **df1 gold** | The trajectory supports a KAQX stop/touch-and-go; df1 combines KSAV-KAQX-KSAV. |
| `aafb20` | **df1 gold** | SFDPS omits a real local KDTO flight around 17:11-17:48. |
| `a25ec5` | **df0 algorithm** | The algorithm misses the KACT touch-and-go and combines KFTW-KACT-KFTW. |
| `a6ea99` | **df1 gold** | Raw data shows the aircraft at KGYY until the 16:36 takeoff; df1 says 15:30. |
| `ad963b` | **df1 gold** | ADS-B shows the aircraft landing at KAPF around 21:42; df1 says 22:43. |
| `acb4b2` | **df0 algorithm** | The algorithm misses the KUXL stop during the ADS-B coverage gap and combines KMSY-KUXL-KBTR. |
| `a9b2c8` | **df1 gold** | SFDPS omits a real local KFMY flight around 17:28-18:37. |

The lower-confidence calls are `a25dca`, `a5a744`, `abfc78`, and `acb4b2`, because the intermediate airport event occurs partly inside an ADS-B coverage gap. The surrounding approach and departure geometry supports the classifications above.

> **Comparison caveat:** this evaluation uses `compare_airports=False`, so a row can be labeled `both` even when df0 and df1 disagree about an intermediate airport. This explains several misleading `df0_only`/`df1_only` combinations.

In [71]:
df.filter(pl.col("icao") == "a81525")

icao,takeoff_time,landing_time,takeoff_airport_ident,landing_airport_ident,takeoff_time_df1,landing_time_df1,takeoff_airport_ident_df1,landing_airport_ident_df1,match_status
str,datetime[ms],datetime[ms],str,str,datetime[ms],datetime[ms],str,str,str
"""a81525""",null,null,null,null,2026-03-01 16:56:00,2026-03-01 17:23:00,"""KAUS""","""KAUS""","""df1_only"""


In [23]:
from data_engineering.flights.flight_type import get_flights
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day


df_flights = get_sfdps_flights_day(date(2026,3,1))

In [24]:
df_flights.filter(pl.col("icao") == "adab24")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""adab24""","""CXK109""","""N980A""",2026-03-01 18:11:00,"""KPIE""",2026-03-01 20:02:00,"""KPIE""",false,true,false,false,"""P28A""","""ARCHER AERO LLC""","""PIPER PA-28-140/150/160/180""",""""""


In [56]:
from datetime import date
from data_engineering.adsb.read_adsb import read_adsb
from airports.airport_lookup import AirportLookup
from flights.evaluation.visual import visualize_flight

def add_airport_ground_level_lines(fig, airport_idents=("KLBX", "KIWS")):
    airport_lookup = AirportLookup()
    for airport_ident in airport_idents:
        airport = airport_lookup.get_Airport_from_airport_ident(airport_ident)
        if airport is None:
            continue
        fig.add_shape(
            type="line",
            x0=0, x1=1, xref="x domain",
            y0=airport.elevation_ft, y1=airport.elevation_ft, yref="y",
            line={"dash": "dot", "color": "#444444"},
        )
        fig.add_annotation(
            x=1, xref="x domain",
            y=airport.elevation_ft, yref="y",
            text=f"{airport.ident} ground: {airport.elevation_ft:,} ft",
            showarrow=False,
            xanchor="right",
            yshift=8,
        )

target_date = date(2026, 3, 1)
df_flights = get_flights(target_date, adsb_src="adsbx")
icao = "a00e86"
df_flights = df_flights.filter(pl.col("icao") == icao)
df_adsb = read_adsb(target_date, icaos=[icao], source="adsbx")
visual = visualize_flight(df_flights, df_adsb, show=False)
add_airport_ground_level_lines(visual)

In [58]:
visual.write_html("a00e86.html", include_plotlyjs=True)

In [59]:
df_flights = get_flights(target_date, adsb_src="adsblol")
df_flights.filter(pl.col("icao") == icao)

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,first_message_time,first_lat,first_lon,first_baro_altitude_ft,first_geom_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,datetime[ms],f64,f64,i64,i64,datetime[ms],f64,f64,i64,i64,bool,bool,bool,bool,str,str,str,str
"""a00e86""","""""","""""",2026-03-01 16:45:20.650,"""KIWS""",2026-03-01 16:58:35.660,"""KT54""",2026-03-01 16:45:18.690,29.808329,-95.668191,250,null,2026-03-01 16:58:35.660,29.47142,-95.731272,2800,null,false,false,false,false,"""C182""","""WINNER PHILIP""","""CESSNA 182 Skylane""","""A1"""
"""a00e86""","""""","""""",2026-03-01 17:39:17.820,"""XS21""",2026-03-01 17:52:51.880,"""KSGR""",2026-03-01 17:42:44.790,29.291134,-95.691026,3775,null,2026-03-01 17:52:51.880,29.516353,-95.654458,2250,null,false,false,false,false,"""C182""","""WINNER PHILIP""","""CESSNA 182 Skylane""","""A1"""
"""a00e86""","""""","""""",2026-03-01 18:04:45.860,"""KAXH""",2026-03-01 18:18:12.290,"""KIWS""",2026-03-01 18:04:45.860,29.497825,-95.575723,2175,null,2026-03-01 18:18:12.290,29.828067,-95.678047,125,null,false,false,false,false,"""C182""","""WINNER PHILIP""","""CESSNA 182 Skylane""","""A1"""
